In [2]:
import torch
import torch.nn as nn
torch.manual_seed(42)

$$
MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2
$$

In [ ]:
# 예측값과 정답값(회귀)
pred = torch.tensor([2.5, 5.0, 7.5])
target = torch.tensor([3.0, 5.0, 7.0])

# pytorch에서 제공하는 MSELoss 함수
mse_buildin = nn.MSELoss()(pred, target)

# 수식기반 수작업
mse_manual = torch.mean((pred - target) ** 2)
print(f"mse_buildin: {mse_buildin}")
print(f"mse_manual: {mse_manual}")

mse_buildin: 0.1666666716337204
mse_manual: 0.1666666716337204


$$
BCE = -\frac{1}{n}\sum_{i=1}^{n}\left[y_i\log(\hat{y}_i)+(1-y_i)\log(1-\hat{y}_i)\right]
$$

In [5]:
# 단일샘플 y = 1일때  식은 -log(y_hat) 단순화되고.. 그래서 정답 확률이 높을수록 손실이 낮아지는 형태가 된다.
y  = torch.tensor([1.0])
yhat_good = torch.tensor([0.9])  # 정답 확률이 높음
yhat_bad = torch.tensor([0.1])   # 정답 확률이 낮음
# 수작업 계산
bce_good_manual = -(y*torch.log(yhat_good) + (1-y)*torch.log(1-yhat_good))
bce_bad_manual = -(y*torch.log(yhat_bad) + (1-y)*torch.log(1-yhat_bad))

# 내장된 BCE
bce_good_builtin = nn.BCELoss()(yhat_good, y)
bce_bad_builtin = nn.BCELoss()(yhat_bad, y)
print(f"bce_good_manual: {bce_good_manual}")
print(f"bce_bad_manual: {bce_bad_manual}")
print(f"bce_good_builtin: {bce_good_builtin}")
print(f"bce_bad_builtin: {bce_bad_builtin}")

bce_good_manual: tensor([0.1054])
bce_bad_manual: tensor([2.3026])
bce_good_builtin: 0.10536054521799088
bce_bad_builtin: 2.3025851249694824


BCE 배치계산

In [6]:
prob_pred = torch.tensor([0.9,0.2,0.4])
label = torch.tensor([1.0,0.0,1.0])

nn.BCELoss()(prob_pred, label)


tensor(0.4149)

CE 다중분류 손실

$$
CE = -\sum_{i=1}^{m} y_i\log(\hat{y}_i)
$$

In [44]:
# 클래스3개 [고양이, 강아지, 토끼]  정답이 강아지 (index =1 )
target_index = torch.tensor([1])
one_hot = torch.tensor([[0.0, 1.0, 0.0]])  # 원-핫 인코딩된 정답

# 좋은 예측과 나쁜 예측
p_good = torch.tensor([0.2, 0.7, 0.1])
p_bad = torch.tensor([0.7, 0.2, 0.1])

# 수작업
ce_good_manual =  -(one_hot*torch.log(p_good)).sum()
ce_bad_manual =  -(one_hot*torch.log(p_bad)).sum()

# torch bce  logits 입력 log(p)
ce_loss =  nn.CrossEntropyLoss()
ce_good_builtin =  ce_loss(torch.log(p_good).unsqueeze(0), target_index)
ce_bad_builtin =  ce_loss(torch.log(p_bad).unsqueeze(0), target_index)

print(ce_good_manual,ce_good_builtin)
print(ce_bad_manual, ce_bad_builtin  )

tensor(0.3567) tensor(0.3567)
tensor(1.6094) tensor(1.6094)


In [54]:
from sklearn.datasets import load_diabetes  # 회귀
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
X, y = load_diabetes(return_X_y= True)

scaler = StandardScaler()
x_train,x_test,y_train,y_test = train_test_split(X,y,random_state=42)
x_train = scaler.fit_transform(x_train)
x_test  = scaler.transform(x_test)

x_train_t = torch.tensor(x_train, dtype=torch.float32)
x_test_t = torch.tensor(x_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train,dtype=torch.float32).unsqueeze(1)
y_test_t = torch.tensor(y_test,dtype=torch.float32).unsqueeze(1)

print(x_train_t.shape)
reg_model  = nn.Sequential(
    nn.Linear(x_train_t.shape[1],64),
    nn.ReLU(),
    nn.Linear(64,1),
)

mse_loss = nn.MSELoss()
optimizer = torch.optim.Adam(reg_model.parameters(), lr=1e-2)  # 0.01

epochs = 300
for _ in range(epochs):
    optimizer.zero_grad()  # 이전가중치 초기화
    pred = reg_model(x_train_t)
    loss = mse_loss(pred, y_train_t)
    loss.backward() # 가중치 찾아서 각 계산과정에 배치
    optimizer.step()  # 각 계산관정의 가중치와 바이어스를 업데이트

with torch.no_grad():
    test_pred = reg_model(x_test_t)
    test_mse = mse_loss(test_pred, y_test_t)

# test_mse  텐서에 들어 있음  출력이나 기타 산술연산이 필요함 꺼내야함.. .item()

print(f'test mse : {test_mse.item():.4f}')

from sklearn.metrics import r2_score
r2_score(y_test_t, test_pred)

torch.Size([331, 10])
test mse : 3127.2034


0.4344704747200012

In [69]:
from sklearn.datasets import load_breast_cancer
# 1. 딥러닝 레이어 sequential에서 입력을 받는 부분을  수정 피처개수
# 2 출력레이어의 활성화 함수로 시그모이드
# 3. 손실함수 nn.BCELoss()
# 추론  with .....  0.5보다 크면 1 그렇지 않으면 0 변경

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
X, y = load_breast_cancer(return_X_y= True)

scaler = StandardScaler()
x_train,x_test,y_train,y_test = train_test_split(X,y,random_state=42)
x_train = scaler.fit_transform(x_train)
x_test  = scaler.transform(x_test)

x_train_t = torch.tensor(x_train, dtype=torch.float32)
x_test_t = torch.tensor(x_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train,dtype=torch.float32).unsqueeze(1)
y_test_t = torch.tensor(y_test,dtype=torch.float32).unsqueeze(1)

print(x_train_t.shape)
reg_model  = nn.Sequential(
    nn.Linear(x_train_t.shape[1],64),
    nn.ReLU(),
    nn.Linear(64,1),
    nn.Sigmoid()
)

mse_loss = nn.BCELoss()
optimizer = torch.optim.Adam(reg_model.parameters(), lr=1e-2)  # 0.01

epochs = 300
for _ in range(epochs):
    optimizer.zero_grad()  # 이전가중치 초기화
    pred = reg_model(x_train_t)
    loss = mse_loss(pred, y_train_t)
    loss.backward() # 가중치 찾아서 각 계산과정에 배치
    optimizer.step()  # 각 계산관정의 가중치와 바이어스를 업데이트

with torch.no_grad():
    test_pred = reg_model(x_test_t)
    test_mse = mse_loss(test_pred, y_test_t)

((test_pred >=0.5).float() == y_test).float().mean()

torch.Size([426, 30])


tensor(0.5282)

CE

In [85]:
from sklearn.datasets import load_iris

# 데이터 변경
# 신경망의 입력데이터 개수
# 신경망의 최종 출력의 뉴런수  3
# 최종출력의 마지막에 활성함수 softmax를 사용 안함 why??  손실함수에서 확률분포로 변경하는 기능 내장

# 손실함수  nn.CrossEntropyLoss()

# 에포크로학습하는 부분까지 완성

# 추론 --> 



from sklearn.datasets import load_iris
# 1. 딥러닝 레이어 sequential에서 입력을 받는 부분을  수정 피처개수
# 2 출력레이어의 활성화 함수로 시그모이드
# 3. 손실함수 nn.BCELoss()
# 추론  with .....  0.5보다 크면 1 그렇지 않으면 0 변경

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
X, y = load_iris(return_X_y= True)

scaler = StandardScaler()
x_train,x_test,y_train,y_test = train_test_split(X,y,random_state=42)
x_train = scaler.fit_transform(x_train)
x_test  = scaler.transform(x_test)

x_train_t = torch.tensor(x_train, dtype=torch.float32)
x_test_t = torch.tensor(x_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train,dtype=torch.long)
y_test_t = torch.tensor(y_test,dtype=torch.long)

reg_model  = nn.Sequential(
    nn.Linear(x_train_t.shape[1],64),
    nn.ReLU(),
    nn.Linear(64,3),
    nn.Softmax()    
)

mse_loss = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(reg_model.parameters(), lr=1e-2)  # 0.01

x_train_t.shape, y_train_t.shape

epochs = 300
for _ in range(epochs):
    optimizer.zero_grad()  # 이전가중치 초기화
    logits = reg_model(x_train_t)
    loss = mse_loss(logits, y_train_t)
    loss.backward() # 가중치 찾아서 각 계산과정에 배치
    optimizer.step()  # 각 계산관정의 가중치와 바이어스를 업데이트

print(f'logits : {logits}')    

with torch.no_grad():
    test_pred = reg_model(x_test_t)
    test_mse = mse_loss(test_pred, y_test_t)

(torch.argmax(test_pred,dim=1) == y_test)



c:\Users\Playdata\.conda\envs\torch_env\Lib\site-packages\torch\nn\modules\module.py:1779: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


logits : tensor([[1.0000e+00, 1.0932e-06, 3.3218e-18],
        [1.0000e+00, 2.5436e-07, 2.4247e-19],
        [1.4042e-09, 5.8842e-08, 1.0000e+00],
        [2.6994e-04, 9.9972e-01, 1.0151e-05],
        [4.4643e-06, 9.9999e-01, 1.0093e-05],
        [9.9997e-01, 2.9279e-05, 3.3753e-17],
        [9.9997e-01, 3.0023e-05, 2.6196e-17],
        [1.5486e-05, 9.9998e-01, 5.6296e-11],
        [1.5846e-05, 5.8224e-02, 9.4176e-01],
        [9.1238e-07, 3.3607e-04, 9.9966e-01],
        [2.0082e-06, 1.0000e+00, 7.0997e-07],
        [3.1640e-08, 2.2997e-05, 9.9998e-01],
        [7.3671e-05, 9.9992e-01, 9.7039e-06],
        [5.4246e-13, 2.0841e-10, 1.0000e+00],
        [6.1205e-04, 9.9483e-01, 4.5623e-03],
        [9.9999e-01, 5.3774e-06, 7.3350e-18],
        [4.8226e-14, 2.5142e-11, 1.0000e+00],
        [1.6804e-05, 9.9998e-01, 7.1824e-09],
        [9.9997e-01, 3.1127e-05, 1.0239e-17],
        [1.0000e+00, 3.8127e-06, 8.7439e-18],
        [9.9999e-01, 7.8201e-06, 4.5350e-16],
        [1.4867e-04, 9.99

tensor([ True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True, False,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True])